# 05 — Offline label coverage EDA

Inspect current repository overlays and the optional pinned Hub corpus cache. This describes annotation coverage, not model performance or human gold: near-gold is an independently checked or reviewed candidate tier. No network, model call, or file write is made. Re-run after label changes; no counts are frozen in this notebook.

In [ ]:
from pathlib import Path
import sys

# Explicitly select this checkout before importing the editable package.
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "src/mailroom_sandbox/notebook.py").is_file())
sys.path.insert(0, str(ROOT / "src"))
from mailroom_sandbox.notebook import bootstrap
ctx = bootstrap(create_dotenv=False, activate_profile=False)
from mailroom_sandbox import label_coverage as lc
from mailroom_sandbox.job.spec import FAMILY_HF_REVISION
from mailroom_sandbox.dojo_source import current
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 30)


## Input identity and source selection

The snapshots are loaded and checked against their manifests on every execution. The dataset pin is the repository's current `FAMILY_HF_REVISION`; the four default/ground-truth train/test shards must already be cached. Their names and exact hashes are reported without machine-specific cache paths. Repository inventory still works when the cache is missing. Actual Dojo selection follows `DOJO_SRC`, then the configured mode and, in auto mode, a matching installed release, selected verified cache, and vendor fallback. The selection shown below is what this process imported, not a claim about a run lock.

In [ ]:
interim, near, overlay_sources = lc.snapshots()
catalog, catalog_sources = lc.catalog()
supplementary, supplementary_sources = lc.catalog("gold-supplementary")
rows, corpus_sources, cache_problem = lc.cached_corpus()
source = current()
print("Pinned dataset revision:", FAMILY_HF_REVISION)
print("Selected Dojo:", {k: getattr(source, k) for k in ("source", "version", "commit", "mode", "reason", "tree_sha256")} if source else "unavailable")
from mailroom_sandbox import scoring_hotfixes
print("SHF-010 enabled:", scoring_hotfixes.enabled("SHF-010"))
from llm_dojo_scoring.schemas import corporate_record_schema, extraction_schemas
from mailroom_sandbox import corporate_extraction
print("Selected schema module SHA256:", {
    "dojo corporate": lc.digest(Path(corporate_record_schema.__file__)),
    "dojo extraction": lc.digest(Path(extraction_schemas.__file__)),
    "sandbox corporate request": lc.digest(Path(corporate_extraction.__file__)),
})
if cache_problem:
    print(cache_problem)
display(pd.DataFrame(overlay_sources + catalog_sources + supplementary_sources + corpus_sources))


## Repository inventory: admitted and not admitted

These are raw records, not effective coverage. Interim `disputed` and catalog `candidate`, `disputed`, or `rejected` records are not admitted as near-gold; a near-gold export is only a candidate overlay and `use_as_truth` is intentionally false. Absence is a documented read, not a populated value. Statuses and bases below retain their original semantics. Unavailable held/pending sources are not inferred.

In [ ]:
inventory = lc.overlay_inventory(interim, "interim") + lc.overlay_inventory(near, "near_gold_export")
inv = pd.DataFrame(inventory)
display(pd.DataFrame(lc.inventory_summary(inventory)))
cat = pd.DataFrame(catalog)
if not cat.empty:
    display(cat.groupby(["doc_class", "subclass", "status", "presence"], dropna=False).size().rename("records").reset_index())
    print("Catalog records with evidence:", sum(bool(x) for x in cat.get("evidence", [])))
    bindings = pd.json_normalize(cat["schema"].dropna()) if "schema" in cat else pd.DataFrame()
    if not bindings.empty:
        display(bindings.value_counts(dropna=False).rename("records").reset_index())
else:
    print("No catalog records available.")

supp = pd.DataFrame(supplementary)
if not supp.empty:
    display(supp.groupby("kind", dropna=False).size().rename("not admitted records").reset_index())


## Effective coverage by class and subclass

Each denominator is the number of corpus documents in the class/subclass for which the field is a registered extraction field, applicable and requested by the runtime schema; label-task fields are excluded. The numerator is the number with a populated effective value. A documented absence is reported separately, and a missing value is a gap. Derived-source fields may score from CUAD/MAUD or subtype labels rather than a direct field: this direct-value view does **not** estimate their derived coverage. Pending registry status is shown, not treated as proof that scoring supports a field. No legacy flat-field denominator is called scored coverage.

The stages use the same overlay application functions as run preparation. Hub values normally win; interim corrections replace their named Hub values; near-gold fills gaps and the enabled SHF-010 rule may replace conflicting interim gate gap fills, retaining adopted original-header values. A text-hash mismatch is not applied. Compare sets of unique document/field pairs: replacements can change values or remove a populated value, so the net change need not equal newly populated pairs.

In [ ]:
if rows:
    stage_rows = lc.stages(rows, interim, near, overlay_sources[0]["sha256"],
                           next(x["sha256"] for x in overlay_sources if x["path"] == "data/labels/gold-overlay/MANIFEST.json"))
    coverage = pd.DataFrame(lc.coverage_rows(stage_rows))
    eligible = coverage[coverage.registered & coverage.applicable & coverage.emittable & coverage.dojo_status.ne("label_task")].copy()
    summary = (eligible.groupby(["stage", "class", "subclass", "field", "dojo_status", "presence"], dropna=False)
               .size().rename("documents").reset_index())
    display(summary)
    rates = eligible.assign(populated=eligible.presence.eq("populated"), documented_absence=eligible.presence.eq("absence"))
    display(rates.groupby(["stage", "class", "subclass", "field"]).agg(
        opportunities=("doc_id", "size"), populated=("populated", "sum"),
        documented_absence=("documented_absence", "sum"), populated_share=("populated", "mean")).reset_index())
    display(eligible.groupby(["stage", "source", "presence"], dropna=False).size().rename("document-field pairs").reset_index())
    pairs = {s: set(zip(g.doc_id, g.field)) for s, g in eligible[eligible.presence == "populated"].groupby("stage")}
    for a, b in (("hub", "interim"), ("interim", "near_gold")):
        before, after = pairs.get(a, set()), pairs.get(b, set())
        print(f"{a} → {b}: {len(before)} → {len(after)} populated pairs; +{len(after-before)} newly populated, -{len(before-after)} no longer populated")
else:
    coverage = eligible = pd.DataFrame()
    print("Corpus comparison unavailable; see cache prerequisite above.")


## Distributions, gaps and schema mismatches

The heatmap uses final-stage populated pairs divided by eligible document–field opportunities. Empty cells have no eligible opportunities. The mismatch table includes runtime output fields outside the registry, registry fields the request cannot emit, and fields the corporate registry marks inapplicable even if requested. Populated labels in those cells are exposed rather than silently counted as eligible.

In [ ]:
if not coverage.empty:
    final = coverage[coverage.stage == "near_gold"]
    gaps = (eligible[eligible.stage == "near_gold"].groupby(["class", "subclass", "field", "presence"])
            .size().unstack(fill_value=0).reset_index())
    display(gaps)
    mismatches = final[(~final.registered) | (~final.emittable) | (~final.applicable)]
    display(mismatches.groupby(["class", "subclass", "field", "dojo_status", "registered", "applicable", "emittable", "presence"], dropna=False).size().rename("documents").reset_index())
    heat = eligible[eligible.stage == "near_gold"].assign(populated=lambda d: d.presence.eq("populated"))
    heat = heat.groupby(["class", "subclass"]).populated.mean().unstack()
    if not heat.empty:
        ax = heat.plot(kind="bar", figsize=(12, 4), ylabel="Populated / eligible pairs", title="Annotation coverage by class and subclass")
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.show()
        fig, ax = plt.subplots(figsize=(max(8, len(heat.columns) * .65), 3))
        image = ax.imshow(heat.fillna(float("nan")), vmin=0, vmax=1, aspect="auto", cmap="Blues")
        ax.set_xticks(range(len(heat.columns)), heat.columns, rotation=90)
        ax.set_yticks(range(len(heat.index)), heat.index)
        fig.colorbar(image, ax=ax, label="Populated share")
        plt.show()


## Evidence examples and drill-down

Set the filters and maximum examples below. Quotes are repository annotations, not newly verified source reads; hashes bind labels to document text. Catalog status and supplementary evidence are displayed without promoting held or disputed material to effective truth. Document text is only available when the pinned corpus cache is complete.

In [ ]:
DOC_ID = ""       # exact document id; empty means all
FIELD = ""        # exact field name; empty means all
MAX_EXAMPLES = 5
examples = inv
if DOC_ID:
    examples = examples[examples.doc_id == DOC_ID]
if FIELD:
    examples = examples[examples.field == FIELD]
display(examples[["source", "doc_id", "class", "subclass", "field", "status", "basis", "presence", "value", "evidence", "doc_sha256"]].head(MAX_EXAMPLES))
if not cat.empty:
    selected = cat
    if DOC_ID:
        selected = selected[selected.doc_id == DOC_ID]
    if FIELD:
        selected = selected[selected.field == FIELD]
    cols = [c for c in ("doc_id", "field", "status", "presence", "value", "evidence", "absent_reason", "content_sha256", "schema") if c in selected]
    display(selected[cols].head(MAX_EXAMPLES))
if DOC_ID and not supp.empty:
    display(supp[supp.doc_id == DOC_ID].head(MAX_EXAMPLES))
if DOC_ID and not coverage.empty:
    selected = coverage[coverage.doc_id == DOC_ID]
    if FIELD:
        selected = selected[selected.field == FIELD]
    display(selected)

# Bounded source view; reading this excerpt does not certify a complete document read.
SOURCE_CHARS = 2000
if DOC_ID and rows:
    import hashlib
    source_rows = [row for row in rows if row["id"] == DOC_ID]
    if source_rows:
        source_row = source_rows[0]
        text = source_row["doc_text"]
        actual_sha = hashlib.sha256(text.encode("utf-8")).hexdigest()
        if source_row.get("content_sha256") and source_row["content_sha256"] != actual_sha:
            raise ValueError("Source text differs from its content hash")
        print({"doc_id": DOC_ID, "source_sha256": actual_sha, "characters": len(text),
               "displayed_range": [0, min(SOURCE_CHARS, len(text))]})
        print(text[:SOURCE_CHARS])
    else:
        print("Document not in the pinned cached corpus.")


## Source-type holds
The source-fit report is a separate bounded audit, not an admitted overlay. It records heuristic instrument groups and exactly which source ranges were inspected. Those groups do not correct corpus subtypes or establish absence. Keep held opportunities in the gap denominator until the corpus owners adjudicate them.

In [ ]:
import json
fit_path = ROOT / "reports/datasets/mailroom-dataset-v9.3-label-repaired/corporate_source_fit_officer_certificate.json"
if fit_path.exists():
    fit = json.loads(fit_path.read_text())
    if fit["dataset_revision"] == FAMILY_HF_REVISION:
        print({"artifact": str(fit_path.relative_to(ROOT)), "sha256": lc.digest(fit_path), "state": fit["state"]})
        display(pd.DataFrame(list(fit["method"]["group_counts_heuristic_all_61"].items()), columns=["heuristic group", "documents"]))
        print("Full source reads:", fit["method"]["full_read_count"])
        display(fit["unresolved_acceptance"])
    else:
        print("Source-fit audit belongs to another dataset revision; it is not current evidence.")
else:
    print("No source-fit audit available in this checkout.")
